In [67]:
import pandas as pd
import networkx as nx
import json
from itertools import combinations
from collections import Counter, defaultdict

In [73]:
cuisine_df = pd.read_csv("https://raw.githubusercontent.com/conwayyao/Recipe-Analysis/master/CuisineAnalyzer/cuisinedata/cuisine_data.csv").drop(columns=["Unnamed: 0"])

def parse(x):
    if pd.isna(x) or x == "[]":
        return pd.NA
    if isinstance(x, list):
        return [i.strip().lower() for i in x]
    return list(i.strip().lower().strip("'\"") for i in x.strip("[]").split(", "))

cuisine_df["ingredient_set"] = cuisine_df["ingredients"].apply(parse)
cuisine_df["course_set"] = cuisine_df["course"].apply(parse)
cuisine_df["cuisine_set"] = cuisine_df["cuisine"].apply(parse)
cuisine_df = (
    cuisine_df
    .drop(columns=["ingredients", "course", "cuisine"])
    .rename(columns={
        "ingredient_set": "ingredients", 
        "course_set": "course", 
        "cuisine_set": "cuisine"
    })
)

cuisine_df = (
    cuisine_df
    .dropna(subset=["ingredients", "course", "cuisine", "totalTimeInSeconds"])
    .drop(columns=["rating"])
    .reset_index(drop=True)
)

def time_group(seconds, group_minutes=10, max_minutes=120):
    minutes = seconds / 60
    if minutes >= max_minutes:
        return "120+ min"
    start = int(minutes // group_minutes) * group_minutes
    end = start + group_minutes - 1
    return f"{start}-{end} min"

cuisine_df["time_group"] = cuisine_df["totalTimeInSeconds"].apply(time_group)
cuisine_df.drop(columns=["totalTimeInSeconds"], inplace=True)

cuisine_df

,id,recipeName,ingredients,course,cuisine,time_group
0,Peanut-butter-fudge-297693,Peanut Butter Fudge,"[country crock® spread, light corn syrup, crea...",[desserts],[american],120+ min
1,Buckeyes-298471,Buckeyes,"[skippy® super chunk® peanut butter, country c...",[desserts],[american],110-119 min
2,Disappearing-buffalo-chicken-dip-297712,Disappearing Buffalo Chicken Dip,"[light mayonnaise, lemon juice, cayenne pepper...",[appetizers],[american],30-39 min
3,Classic-macaroni-salad-304692,Classic Macaroni Salad,"[elbow macaroni, hellmann' or best food real m...","[salads, side dishes]",[american],20-29 min
4,Classic-coleslaw-303481,Classic Coleslaw,"[hellmann' or best food real mayonnais, lemon ...","[salads, side dishes]","[american, southern & soul food]",10-19 min
...,...,...,...,...,...,...
7029,Thai-Coconut-Soup-MyRecipes-246201,Thai Coconut Soup,"[chicken broth, coconut milk, fish sauce, fres...",[soups],"[thai, asian]",40-49 min
7030,Thai-Coconut-Shrimp-Soup-901927,Thai Coconut Shrimp Soup,"[olive oil, large shrimp, carrots, garlic, gin...",[soups],[thai],20-29 min
7031,Spicy-Thai-Lobster-Soup-My-Recipes,Spicy Thai Lobster Soup,"[lobster, vegetable oil, asian, fish, lime rin...","[main dishes, soups]","[asian, thai]",50-59 min
7032,Thai-Green-Papaya-Salad-_Som-Tam_-1266284,Thai Green Papaya Salad (Som Tam),"[fresh lime juice, palm sugar, fish sauce, gar...",[main dishes],"[barbecue, thai]",20-29 min


In [74]:
courses = dict()
cuisines = dict()
tgs = dict()
for _, row in cuisine_df.iterrows():
    for course in row["course"]:
        courses[course] = courses.get(course, 0) + 1
    for cuisine in row["cuisine"]:
        cuisines[cuisine] = cuisines.get(cuisine, 0) + 1
    tgs[row["time_group"]] = tgs.get(row["time_group"], 0) + 1

courses, cuisines, tgs

({'desserts': 611,
  'appetizers': 579,
  'salads': 532,
  'side dishes': 619,
  'condiments and sauces': 290,
  'main dishes': 3523,
  'lunch and snacks': 450,
  'breakfast and brunch': 244,
  'beverages': 175,
  'afternoon tea': 20,
  'breads': 282,
  'soups': 540,
  'cocktails': 58},
 {'american': 942,
  'southern & soul food': 310,
  'french': 323,
  'mexican': 374,
  'southwestern': 203,
  'asian': 1239,
  'kid-friendly': 89,
  'barbecue': 259,
  'cajun & creole': 449,
  'cuban': 221,
  'spanish': 330,
  'irish': 200,
  'italian': 405,
  'mediterranean': 267,
  'hawaiian': 173,
  'chinese': 223,
  'thai': 479,
  'japanese': 378,
  'indian': 257,
  'greek': 539,
  'english': 27,
  'german': 239,
  'hungarian': 297,
  'portuguese': 249,
  'moroccan': 162,
  'swedish': 194},
 {'120+ min': 610,
  '110-119 min': 29,
  '30-39 min': 1227,
  '20-29 min': 1082,
  '10-19 min': 604,
  '80-89 min': 260,
  '40-49 min': 1053,
  '100-109 min': 176,
  '60-69 min': 623,
  '50-59 min': 685,
  '0-9 

In [75]:
# node: ingredient = (count of recipes, {cuisine: count of recipes}, {course: count of recipes})
# edge: ingredient1 -> ingredient2 = (count of co-occurrence in recipes, {cuisine: count of co-occurrence}, {course: count of co-occurrence})

node_recipe_count = Counter()
node_cuisine_count = defaultdict(Counter)
node_course_count = defaultdict(Counter)
node_time_group_count = defaultdict(Counter)

edge_recipe_count = Counter()
edge_cuisine_count = defaultdict(Counter)
edge_course_count = defaultdict(Counter)
edge_time_group_count = defaultdict(Counter)

for idx, row in cuisine_df.iterrows():
    print(f"Processing recipe {idx+1}/{len(cuisine_df)}", end="\r")

    ingredients = row["ingredients"]
    cuisines = row["cuisine"]
    courses = row["course"]
    time_group = row["time_group"]

    if not isinstance(ingredients, list):
        continue

    ingredients = list(dict.fromkeys(ingredients))

    for ingredient in ingredients:
        node_recipe_count[ingredient] += 1

        if isinstance(cuisines, list):
            for cuisine in cuisines:
                node_cuisine_count[ingredient][cuisine] += 1

        if isinstance(courses, list):
            for course in courses:
                node_course_count[ingredient][course] += 1

        if pd.notna(time_group):
            node_time_group_count[ingredient][time_group] += 1

    for ingredient1, ingredient2 in combinations(ingredients, 2):
        edge = tuple(sorted((ingredient1, ingredient2)))

        edge_recipe_count[edge] += 1

        if isinstance(cuisines, list):
            for cuisine in cuisines:
                edge_cuisine_count[edge][cuisine] += 1

        if isinstance(courses, list):
            for course in courses:
                edge_course_count[edge][course] += 1

        if pd.notna(time_group):
            edge_time_group_count[edge][time_group] += 1

In [76]:
network = nx.Graph()

min_node_count = 3
min_edge_count = 3

valid_ingredients = {
    ingredient
    for ingredient, recipe_count in node_recipe_count.items()
    if recipe_count >= min_node_count
}

for ingredient in valid_ingredients:
    network.add_node(
        ingredient,
        recipe_count=node_recipe_count[ingredient],
        cuisines=dict(node_cuisine_count[ingredient]),
        courses=dict(node_course_count[ingredient]),
        time_groups=dict(node_time_group_count[ingredient])
    )

for (ingredient1, ingredient2), recipe_count in edge_recipe_count.items():
    if recipe_count < min_edge_count:
        continue
    if ingredient1 not in valid_ingredients or ingredient2 not in valid_ingredients:
        continue
    network.add_edge(
        ingredient1, ingredient2,
        recipe_count=recipe_count,
        cuisines=dict(edge_cuisine_count[(ingredient1, ingredient2)]),
        courses=dict(edge_course_count[(ingredient1, ingredient2)]),
        time_groups=dict(edge_time_group_count[(ingredient1, ingredient2)])
    )

network.number_of_nodes(), network.number_of_edges()

(1892, 29663)

In [77]:
network_graphml = network.copy()

for _, attrs in network_graphml.nodes(data=True):
    for key, value in list(attrs.items()):
        if isinstance(value, dict):
            attrs[key] = json.dumps(value)

for _, _, attrs in network_graphml.edges(data=True):
    for key, value in list(attrs.items()):
        if isinstance(value, dict):
            attrs[key] = json.dumps(value)

nx.write_graphml(network_graphml, "ingredient_network.graphml")